# Polar and Cartesian images

`pypft.cartesian_to_polar`/`pypft.polar_to_cartesian` resample an ordinary
image onto (and back off of) a *uniform* polar grid, via `cv2.warpPolar`.

**This is not the discrete Hankel transform's own sampling grid** -- that
grid is order-dependent and non-uniform, and comes later
(`pypft.grid.PolarGrid`). These two functions exist because `warpPolar` is
the natural first illustration of what "polar" means for an image, not
because it feeds the transform.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import pypft

## A synthetic test image

A handful of angular wedges plus a few concentric rings make both axes of
the polar resampling easy to see at a glance.

In [ ]:
size = 256
center = size // 2
rows, cols = np.mgrid[0:size, 0:size]
dx, dy = (cols - center).astype(float), (rows - center).astype(float)
r = np.hypot(dx, dy)
cv_angle = np.arctan2(dy, dx)  # OpenCV's own angle convention -- see below

wedges = (np.sin(4 * cv_angle) > 0).astype(float)
rings = (np.sin(r / 6) > 0).astype(float)
image = 0.5 * wedges + 0.5 * rings
image[r > 0.95 * size / 2] = 0.0

plt.imshow(image, cmap="gray")
plt.title("Cartesian test image")
plt.show()

## To polar and back

`cartesian_to_polar` returns a `(radial, angular)` array -- PyPFT's own axis
layout (`pypft.Axis`), the opposite of `cv2.warpPolar`'s native
`(angular, radial)`. Its angular axis is also *centered*: index
`n_angular // 2` holds angle `0`, not index `0`.

In [ ]:
n_radial, n_angular = 128, 96
polar = pypft.cartesian_to_polar(image, n_radial, n_angular)
reconstructed = pypft.polar_to_cartesian(polar, size, size)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(image, cmap="gray")
axes[0].set_title("original")
axes[1].imshow(polar.T, cmap="gray", aspect="auto")
axes[1].set_title("polar (radial down, angular across)")
axes[2].imshow(reconstructed, cmap="gray")
axes[2].set_title("reconstructed")
fig.tight_layout()
plt.show()

Away from the disk's boundary (where `warpPolar` has no data to sample from
in the first place), the round trip is close to the original:

In [ ]:
disk = r < 0.85 * size / 2
float(np.abs(image[disk] - reconstructed[disk]).mean())

## The angular origin

`cv2.warpPolar` measures its angle directly on image coordinates -- i.e.
`atan2(row - center_y, col - center_x)`, with no flip for `row` growing
downward. Rotating from the positive-`x` axis towards the positive-`y` axis
(downward on screen) is therefore the *positive* angular direction: counter-
clockwise in image coordinates, but clockwise as the image is drawn.

A single wedge at a known angle shows where it lands in the centered polar
array:

In [ ]:
phi0 = np.pi / 2  # OpenCV's own angle convention, not the mathematical one
wedge = (np.abs((cv_angle - phi0 + np.pi) % (2 * np.pi) - np.pi) < np.deg2rad(5)) & (
    r < 0.9 * size / 2
)
wedge_polar = pypft.cartesian_to_polar(wedge.astype(float), n_radial, n_angular)

measured_index = int(np.argmax(wedge_polar.sum(axis=0)))
predicted_index = (n_angular // 2 + round(phi0 * n_angular / (2 * np.pi))) % n_angular
measured_index, predicted_index

`measured_index` and `predicted_index` agree: index `n_angular // 2 +
round(phi0 * n_angular / (2 * pi))`, taken modulo `n_angular`, is exactly
where a wedge at OpenCV-angle `phi0` ends up once the angular axis has been
centered.